# SignalDesk Weekly Health Check

**Who this is for:** the product teammate who asked *"what's working, what looks suspicious, what should we look at next?"*

**Data:** `data/product_usage_events.csv` — daily usage rows for three AI-assisted workflows (Lead summary, Reply draft, Feedback clustering). See `../ds-intern-challenge/domain-packet.md` for the full context and the field caveats (`sessions`, `completed`, `accepted_output`, `flagged_for_review`, `avg_minutes_saved`, `median_confidence` all have gotchas — read those before trusting any of them at face value).

**Plan for this notebook:**
1. Load the data and look at it raw, before touching anything.
2. Clean what's safely fixable; flag (don't silently fix) what isn't.
3. Compare the three workflows on **rate** metrics — "useful" = high acceptance rate, high avg minutes saved, high rating.
4. Look at before vs. after the prompt change mentioned in the notes column.
5. Two *separate* kinds of suspicious rows: (a) confidence moving opposite to rating/flag-rate, (b) volume spikes / duplicates.
6. Write up findings and what to look at next.

**On modeling:** we're deliberately using rules/thresholds here, not an ML model. With ~40 rows across only 7 days, there's no real train/test split and any "model" would just memorize the week. A heuristic you can explain in one sentence is stronger signal for this challenge than a fitted model would be — see `rubric.md`. Worth a mention in "what's next", not worth building now.

## 0. Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

DATA_PATH = "data/product_usage_events.csv"
PROMPT_CHANGE_DATE = "2026-08-04"  # from notes: "new prompt version started"

## 1. Load and look at the raw data

Don't clean anything yet — just look. What columns are there, what types did pandas infer, does anything jump out (duplicates, weird casing, missing values, obviously-wrong numbers)?

In [2]:
df = pd.read_csv(DATA_PATH)
df

,date,team,workflow,source,sessions,completed,accepted_output,flagged_for_review,avg_minutes_saved,median_confidence,user_rating,notes
0,2026-08-01,Sales,Lead summary,email,42,35,29,3,8.5,0.74,4.1,normal day
1,2026-08-01,Sales,Lead summary,manual,18,12,8,2,6.0,0.61,3.8,normal day
2,2026-08-01,Support,Reply draft,queue,55,48,39,6,4.5,0.82,4.0,normal day
3,2026-08-01,Support,Reply draft,manual,11,8,5,1,3.0,0.68,NaN,missing rating
4,2026-08-01,Product,Feedback clustering,csv upload,12,9,6,2,14.0,0.59,3.6,small sample
5,2026-08-01,Product,Feedback clustering,manual,5,4,3,1,11.0,0.55,3.5,small sample
6,2026-08-02,Sales,Lead summary,email,46,38,31,3,8.2,0.76,4.2,normal day
7,2026-08-02,Sales,Lead summary,manual,20,14,9,2,6.4,0.63,3.9,normal day
8,2026-08-02,Support,Reply draft,queue,61,52,41,7,4.2,0.83,4.1,normal day
9,2026-08-02,Support,Reply draft,manual,12,9,6,1,3.2,0.69,3.7,normal day


In [3]:
print(df.dtypes)
print()
print("team values:", sorted(df["team"].unique()))
print("median_confidence has literal 'n/a' strings:", (df["median_confidence"] == "n/a").sum())
print("exact duplicate rows (ignoring notes):",
      df.drop(columns="notes").duplicated().sum())
print("missing user_rating:", df["user_rating"].isna().sum())

date                      str
team                      str
workflow                  str
source                    str
sessions                int64
completed               int64
accepted_output         int64
flagged_for_review      int64
avg_minutes_saved     float64
median_confidence     float64
user_rating           float64
notes                     str
dtype: object

team values: ['Product', 'Sales', 'Support', 'product']
median_confidence has literal 'n/a' strings: 0
exact duplicate rows (ignoring notes): 1
missing user_rating: 1


**Notes on what you noticed here:**

- `team` has a casing inconsistency (`product` vs `Product`).
- `median_confidence` loaded as an `object` column, not numeric — because of the literal string `"n/a"` mixed in with real numbers.
- One exact duplicate row (identical everywhere except `notes`).
- One row missing `user_rating`.

## 2. Clean

Things worth deciding on:

- inconsistent `team` casing → safe to normalize.
- an exact duplicate row (same metrics, different `notes` text) → drop one copy; note it, don't silently swallow it.
- `median_confidence` blank vs. literal `"n/a"` → both become a real missing value (`NaN`), not treated as different things.
- missing `user_rating` → leave as missing, don't impute.

In [4]:
df_clean = df.copy()
cleaning_log = []

# --- team casing ---
before = df_clean["team"]
df_clean["team"] = df_clean["team"].str.title()
n_fixed = (before != df_clean["team"]).sum()
if n_fixed:
    cleaning_log.append(f"Normalized team casing on {n_fixed} row(s).")

# --- median_confidence: "n/a" string and blanks -> real NaN, then numeric ---
df_clean["median_confidence"] = pd.to_numeric(
    df_clean["median_confidence"].replace("n/a", pd.NA), errors="coerce"
)
n_missing_conf = df_clean["median_confidence"].isna().sum()
cleaning_log.append(f"{n_missing_conf} row(s) have no median_confidence (left as missing, not imputed).")

# --- user_rating: just count what's missing ---
n_missing_rating = df_clean["user_rating"].isna().sum()
cleaning_log.append(f"{n_missing_rating} row(s) have no user_rating (left as missing, not imputed).")

# --- exact duplicates (same everything except notes) ---
dup_mask = df_clean.drop(columns="notes").duplicated(keep="first")
n_dupes = dup_mask.sum()
if n_dupes:
    cleaning_log.append(
        f"Dropped {n_dupes} exact duplicate row(s) (identical metrics, different "
        f"'notes' text — looks like a double export, not two real events)."
    )
df_clean = df_clean.loc[~dup_mask].reset_index(drop=True)

for entry in cleaning_log:
    print("-", entry)
print(f"\n{len(df)} raw rows -> {len(df_clean)} rows after cleaning.")

- Normalized team casing on 1 row(s).
- 1 row(s) have no median_confidence (left as missing, not imputed).
- 1 row(s) have no user_rating (left as missing, not imputed).
- Dropped 1 exact duplicate row(s) (identical metrics, different 'notes' text — looks like a double export, not two real events).

41 raw rows -> 40 rows after cleaning.


## 3. Compare workflows — finalized definition of "useful"

**Useful = high acceptance rate + high avg minutes saved + high rating.**

Rates, not raw counts — `accepted_output / completed`, not raw `accepted_output` — so a high-traffic day doesn't look "more useful" just because it's bigger.

In [5]:
def summarize(group: pd.DataFrame) -> pd.Series:
    sessions = group["sessions"].sum()
    completed = group["completed"].sum()
    accepted = group["accepted_output"].sum()
    flagged = group["flagged_for_review"].sum()
    return pd.Series({
        "sessions": sessions,
        "completion_rate": completed / sessions if sessions else None,
        "acceptance_rate": accepted / completed if completed else None,
        "flag_rate": flagged / sessions if sessions else None,
        # weighted by completed / sessions so a tiny-volume day doesn't skew the average
        "avg_minutes_saved": (group["avg_minutes_saved"] * group["completed"]).sum() / completed if completed else None,
        "avg_confidence": (group["median_confidence"] * group["sessions"]).sum() / group.loc[group["median_confidence"].notna(), "sessions"].sum()
            if group["median_confidence"].notna().any() else None,
        "avg_rating": (group["user_rating"] * group["sessions"]).sum() / group.loc[group["user_rating"].notna(), "sessions"].sum()
            if group["user_rating"].notna().any() else None,
    })

by_workflow = df_clean.groupby("workflow").apply(summarize, include_groups=False).round(2)
by_workflow

,sessions,completion_rate,acceptance_rate,flag_rate,avg_minutes_saved,avg_confidence,avg_rating
workflow,,,,,,,
Feedback clustering,207.0,0.67,0.66,0.13,12.99,0.62,3.73
Lead summary,590.0,0.81,0.82,0.06,9.03,0.79,4.31
Reply draft,510.0,0.81,0.76,0.14,3.93,0.83,3.94


In [6]:
# per-source breakdown, since sources aren't directly comparable
by_workflow_source = df_clean.groupby(["workflow", "source"]).apply(summarize, include_groups=False).round(2)
by_workflow_source

sessions  completion_rate  acceptance_rate  \
workflow            source                                                   
Feedback clustering csv upload     149.0             0.64             0.68   
                    manual          58.0             0.72             0.62   
Lead summary        email          450.0             0.84             0.85   
                    manual         140.0             0.70             0.70   
Reply draft         manual          84.0             0.74             0.66   
                    queue          426.0             0.82             0.77   

                                flag_rate  avg_minutes_saved  avg_confidence  \
workflow            source                                                     
Feedback clustering csv upload       0.13              13.85            0.64   
                    manual           0.12              11.03            0.56   
Lead summary        email            0.05               9.70            0.83   
                    manual           0.09               6.46            0.65   
Reply draft         manual           0.10               3.36            0.71   
                    queue            0.14               4.03            0.86   

                                avg_rating  
workflow            source                  
Feedback clustering csv upload        3.81  
                    manual            3.54  
Lead summary        email             4.43  
                    manual            3.93  
Reply draft         manual            3.83  
                    queue             3.96

## 4. Before vs. after the prompt change

Compare rate metrics before vs. after `PROMPT_CHANGE_DATE`, per workflow. Support/Reply draft also has a review-policy change on 2026-08-07 — its "after" window reflects two overlapping changes, not just the new prompt, so read that row with extra caution.

In [7]:
before = df_clean[df_clean["date"] < PROMPT_CHANGE_DATE].groupby("workflow").apply(summarize, include_groups=False)
after = df_clean[df_clean["date"] >= PROMPT_CHANGE_DATE].groupby("workflow").apply(summarize, include_groups=False)

before_after = pd.concat(
    {"before": before[["acceptance_rate", "flag_rate", "avg_rating"]],
     "after": after[["acceptance_rate", "flag_rate", "avg_rating"]]},
    axis=1,
).round(2)
before_after

before                                after  \
                    acceptance_rate flag_rate avg_rating acceptance_rate   
workflow                                                                   
Feedback clustering            0.67      0.14       3.64            0.66   
Lead summary                   0.77      0.08       4.07            0.85   
Reply draft                    0.77      0.11       4.00            0.75   

                                          
                    flag_rate avg_rating  
workflow                                  
Feedback clustering      0.12       3.77  
Lead summary             0.05       4.44  
Reply draft              0.15       3.91

## 5a. Suspicious rows — confidence vs. outcome

**Flag rows where `median_confidence` is in the top quartile for its workflow *and* either `user_rating` is low or the `flagged_for_review` rate is high.** That's the model being more "sure" of itself while humans push back more — a genuine contradiction, unlike accepted+flagged both being nonzero.

In [8]:
def flag_confidence_divergence(group: pd.DataFrame) -> pd.Series:
    conf = group["median_confidence"]
    if conf.notna().sum() < 3:
        return pd.Series(False, index=group.index)
    conf_hi = conf.quantile(0.75)
    flag_rate = group["flagged_for_review"] / group["sessions"]
    bad_rating = group["user_rating"] < 3.0
    high_flags = flag_rate >= 0.25
    return (conf >= conf_hi) & (bad_rating.fillna(False) | high_flags)

df_clean["confidence_divergence_flag"] = (
    df_clean.groupby("workflow", group_keys=False).apply(flag_confidence_divergence, include_groups=False)
)

cols = ["date", "team", "workflow", "source", "median_confidence", "user_rating", "flagged_for_review", "sessions", "notes"]
df_clean.loc[df_clean["confidence_divergence_flag"], cols]

,date,team,workflow,source,median_confidence,user_rating,flagged_for_review,sessions,notes
37,2026-08-07,Support,Reply draft,queue,0.91,2.1,12,30,review policy changed mid-day


## 5b. Suspicious rows — volume spikes & duplicates (data-integrity, kept separate from 5a)

In [9]:
def flag_volume_spike(group: pd.DataFrame) -> pd.Series:
    if len(group) < 3:
        return pd.Series(False, index=group.index)
    med = group["sessions"].median()
    return group["sessions"] >= 2.0 * med if med > 0 else pd.Series(False, index=group.index)

df_clean["volume_spike_flag"] = (
    df_clean.groupby(["team", "workflow", "source"], group_keys=False)
    .apply(flag_volume_spike, include_groups=False)
)

print(f"Duplicate rows already dropped during cleaning: {n_dupes}")
df_clean.loc[df_clean["volume_spike_flag"], ["date", "team", "workflow", "source", "sessions", "notes"]]

Duplicate rows already dropped during cleaning: 1


,date,team,workflow,source,sessions,notes
24,2026-08-05,Sales,Lead summary,email,140,traffic spike from demo account


## 6. Findings

**Lead summary (Sales) is the most useful workflow right now.** It has the highest acceptance rate and highest average rating of the three (Section 3), even before accounting for the demo-account spike that inflates one day of its volume.

**`median_confidence` is the metric to trust least.** It's model-reported, not a correctness signal, and Section 5a surfaces a concrete case that proves it: on 2026-08-07 (Support/Reply draft), confidence hit its weekly high the same day user rating hit its weekly low and review flags spiked — moving the *opposite* direction from the signals that actually reflect quality.

**The prompt change had a mixed, workflow-dependent effect** (Section 4). Lead summary improved on every rate metric after 08-04. Reply draft looks worse after, but that window is confounded by a same-day review-policy change on 08-07 — so its dip is uncertain, not clearly attributable to the prompt.

**Next to investigate:** the Aug 5 demo-account spike (segment it out before trusting Sales/Lead summary averages), the export pipeline that produced the duplicate row, and the Aug 7 Support/queue day specifically, before rolling the review-policy change out further.

**A fitted model isn't warranted yet.** ~40 rows spanning 7 days leaves no real train/test split, so a model would just memorize the week. Worth revisiting once this export spans weeks or months instead of days.